### **Create Balanced Cohort**


This function constructs a **balanced, temporally standardized fMRI dataset** for binary classification (e.g., case vs. control) from UK Biobank data. It ensures **fair class representation**, **temporal alignment across subjects**, and **compatible preprocessing** with multi-dataset studies (e.g., ABIDE + UKB).


- **Balanced Sampling**: Optionally undersamples the majority class to match case/control counts.
- **Temporal Standardization**: Resamples all fMRI time series to a **universal temporal grid** (default: 300s duration at 2.0s TR → 150 timepoints).
- **Post-Resample Z-Scoring**: Applies **per-subject, per-ROI standardization** *after* resampling—critical for avoiding interpolation artifacts and ensuring distributional compatibility.
- **Phenotype Alignment**: Safely aligns fMRI and phenotype data by `eid` (UKB subject ID), with error checking.



In [3]:
import pandas as pd

# Load data
pheno = pd.read_csv('/home/jaizor/jaizor/xtra/notebooks/UKBB/dataset/ukbb_pheno.csv')
print(f"✓ Loaded phenotype: {pheno.shape}")

# Exclude non-binary columns: 'eid' (string), 'Age' (continuous)
binary_cols = [col for col in pheno.columns if col not in ['eid', 'Age']]

# Count number of 1s (cases) for each binary column
case_counts = pheno[binary_cols].sum().astype(int)

# Optional: Also compute prevalence (%)
prevalence_pct = (case_counts / len(pheno)) * 100

# Combine into a summary DataFrame
summary = pd.DataFrame({
    'n_cases': case_counts,
    'prevalence_%': prevalence_pct.round(2)
})

# Sort by number of cases (descending)
summary = summary.sort_values('n_cases', ascending=False)

# Display top 30
print(f"\n📊 Total subjects: {len(pheno):,}")
print(f"🧾 Binary features: {len(binary_cols)}")
print("\nTop Most Frequent Conditions:")
print(summary.head(100).to_string())


✓ Loaded phenotype: (16340, 55)

📊 Total subjects: 16,340
🧾 Binary features: 53

Top Most Frequent Conditions:
                                                             n_cases  prevalence_%
neuro_healthy_clean                                            12376         75.74
neuro_healthy                                                  10918         66.82
Chronic_Musculoskeletal                                        10169         62.23
Chronic_Digestive                                              10017         61.30
Chronic_Respiratory                                             8915         54.56
Chronic_Cardiovascular                                          8325         50.95
Chronic_Genitourinary                                           8313         50.88
Sex                                                             7613         46.59
Chronic_Skin_Subcutaneous                                       6963         42.61
neuro_pathology                                            

In [ ]:
import numpy as np
import pandas as pd
import os
from data import create_balanced_cohort

# ==============================
# DEFINE YOUR TARGET CONDITIONS
# ==============================
TARGET_CONDITIONS = [
    'neuro_pathology',
    'ICD_G40_Epilepsy',
    'ICD_F33_Recurrent_Depressive',
    'Psychopathology_Organic_Mental_Disorder',
    'NervousSystem_Other_Neuro',
    'NervousSystem_Multiple_Sclerosis_Other_Demyelinating',
    'NervousSystem_Dementia_Developmental',
    'ICD_F31_Bipolar',
    'ICD_G20_Parkinsons'
]

CONTROL_COLUMN = 'neuro_healthy'
RANDOM_SEED = 42
OUTPUT_DIR = 'dataset/new'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Shared resampling config
RESAMPLE_KWARGS = {
    'resample_data': True,
    'original_tr': 0.735,      # UKB's actual TR
    'target_tr': 2.0,          # Standard TR
    'target_duration': 300.0   # 5 minutes
}

# ==============================
# LOOP OVER EACH CONDITION
# ==============================
for TARGET_COLUMN in TARGET_CONDITIONS:
    print(f"\nProcessing: {TARGET_COLUMN}")
    
    FMRI_OUT = os.path.join(OUTPUT_DIR, f'fmri_{TARGET_COLUMN}.npz')
    PHENO_OUT = os.path.join(OUTPUT_DIR, f'pheno_{TARGET_COLUMN}.csv')
    
    try:
        fMRI, labels, subject_ids, pheno_final = create_balanced_cohort(
            fmri_path='dataset/ukbb_features_zscored.npz',
            pheno_path='dataset/ukbb_pheno.csv',
            target_column=TARGET_COLUMN,
            control_column=CONTROL_COLUMN,
            balance=True,
            random_seed=RANDOM_SEED,
            **RESAMPLE_KWARGS
        )

        print(f"✅ Cohort size: {len(subject_ids)} "
              f"({int(labels.sum())} cases, {len(labels) - int(labels.sum())} controls)")

        # Ensure 'eid' is a column
        if pheno_final.index.name == 'eid' or 'eid' not in pheno_final.columns:
            pheno_final = pheno_final.reset_index()
        
        # Safety checks
        assert 'eid' in pheno_final.columns, "eid must be a column!"
        assert list(pheno_final['eid']) == list(subject_ids), "eid mismatch!"

        # Save
        np.savez_compressed(FMRI_OUT, data=fMRI, subject_ids=subject_ids)
        pheno_final.to_csv(PHENO_OUT, index=False)

        print(f"💾 Saved: fMRI={FMRI_OUT}, Pheno={PHENO_OUT} ({pheno_final.shape})")

    except Exception as e:
        print(f"❌ Failed for {TARGET_COLUMN}: {e}")
        continue

Using device: cuda

Processing: neuro_pathology
✓ Loaded phenotype: (16340, 54)
✓ Loaded fMRI: (16340, 490, 414)
  Subjects: 16340
  Timepoints: 490
  Brain regions: 414
Found 3964 cases, 10918 eligible controls
🔄 Resampling to universal standard:
   Original: TR=0.735s, Duration=360.1s
   Target:   TR=2.0s, Duration=300.0s (5 min)
   Output:   150 timepoints
✅ Resampled shape: (7928, 150, 414)
📊 Standardizing after resampling (per subject, per ROI)...
✅ Standardization complete.
✓ Final dataset: 7928 subjects (3964 cases, 3964 controls)
✅ Cohort size: 7928 (3964 cases, 3964 controls)
💾 Saved: fMRI=dataset/new/fmri_neuro_pathology.npz, Pheno=dataset/new/pheno_neuro_pathology.csv ((7928, 56))

Processing: ICD_G40_Epilepsy
✓ Loaded phenotype: (16340, 54)
✓ Loaded fMRI: (16340, 490, 414)
  Subjects: 16340
  Timepoints: 490
  Brain regions: 414
Found 168 cases, 10918 eligible controls
🔄 Resampling to universal standard:
   Original: TR=0.735s, Duration=360.1s
   Target:   TR=2.0s, Duration